# Bronze — deterministic synthetic leases

Creates **`dim_property`** (with **`ecosystem_segment`**, **`brand_line`**, **`metro_cluster`**, **`flagship_style`**, etc.), **`dim_unit`**, and **`fact_lease_episode`** as Parquet mirrors of a Databricks medallion landing zone — same generators as local `scripts/run_local_pipeline.py`.

**Databricks setup:** attach this repo (`python/` folder contains `cs_portfolio`). Set `PORTFOLIO_BASE_PATH` (for example `/dbfs/FileStore/portfolio_demo` or `/Volumes/catalog/schema/vol`) before running imports.

In [ ]:
%pip install -q pandas pyarrow numpy scikit-learn mlflow joblib
import os, sys

# Repo root containing `python/cs_portfolio` — adjust after importing this notebook into a Databricks Repo
REPO = os.environ.get("PORTFOLIO_REPO_ROOT", "../..")
sys.path.insert(0, f"{REPO}/python")

if os.environ.get("PORTFOLIO_BASE_PATH") is None and "dbutils" in globals():
    dbutils.widgets.text("base_path", "")  # type: ignore
    _bp = dbutils.widgets.get("base_path")  # type: ignore
    if _bp:
        os.environ["PORTFOLIO_BASE_PATH"] = _bp

from cs_portfolio.synthetic import SyntheticConfig, run_synthetic_to_bronze

run_synthetic_to_bronze(SyntheticConfig(random_seed=42))

## Optional — persist as Delta (`bronze_fact_lease_episode`)

Uncomment on a Spark cluster where Delta is enabled. Paths should point at your UC volume or legacy `abfss://` location.

In [ ]:
# from pyspark.sql import SparkSession
# spark = SparkSession.builder.getOrCreate()
# base = spark.conf.get("spark.databricks.workspaceUrl", "local")
# df = spark.read.parquet(...)  # set to pandas output converted via spark.createDataFrame or read files
# df.write.format("delta").mode("overwrite").saveAsTable("dev_portfolio.bronze.fact_lease_episode")